<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/neural_importance_sampling_asimov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural importance-sampling Asimov construction

The YAML supplies the design points, defensive mixture $\epsilon$, pilot size, proposal flow, and target quadrature size. This notebook trains every required model through the native package and reports validation and ESS.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'examples' / 'lhc_analysis'))


In [ ]:
from generate_distributions import generate
from hnsbi import Project

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
generate(EXAMPLE / 'data', signal_events=12_000, background_events=30_000, reference_events=40_000)
project = Project.load(EXAMPLE / 'analysis.yaml')
reference_artifacts = project.train_reference()
reference = reference_artifacts.training.flow
ratios = project.train_ratios(reference, normalization_events=40_000, seed=20260729)
systematic_training = project.train_systematics()
runtime_systematics = project.build_runtime_systematics(systematic_training)


## Train and validate the proposal

The defensive density is

$$
g_\epsilon(x)=(1-\epsilon)g_\psi(x)+\epsilon q_\phi(x).
$$

All proposal diagnostics and ONNX parity results are included in the returned artifact bundle.

In [ ]:
nis = project.train_nis_asimov(reference=reference, ratios=ratios.evaluators, truth_point={'mu': 1.0, 'response': 0.0, 'resolution': 0.0, 'theory': 0.0}, asimov_point={'mu': 1.0, 'response': 0.0, 'resolution': 0.0, 'theory': 0.0}, systematics=runtime_systematics)
print('raw count:', nis.asimov.raw_count)
print('ESS:', nis.asimov.ess)
print('validation:', nis.validation_provenance)
print('workspace-ready arrays:', nis.asimov_array_paths)
